# Data Preparation & Feature Engineering
This notebook covers data loading, exploration, location/user/temporal feature extraction, and data cleanup for tourism reviews.

In [3]:
import pandas as pd
import re
import sys
import subprocess

df = pd.read_csv("Reviews.csv", encoding='latin1')
# Save a working copy for all processing
df.to_csv("Processed_Reviews.csv", index=False)
# All further processing should use Processed_Reviews.csv
df = pd.read_csv("Processed_Reviews.csv")

# Basic inspection
print("Dataset Overview:")
print(f"  Shape: {df.shape}")
print(f"  Columns: {df.columns.tolist()}")
print(f"\n Data Types:")
print(df.dtypes)
print(f"\n Missing Values:")
print(df.isnull().sum())
print(f"\n Duplicates: {df.duplicated().sum()}")
print(f"\n Rating Distribution:")
print(df["Rating"].value_counts().sort_index())
print(f"\n Top 10 Cities:")
print(df["Located_City"].value_counts().head(10))

Dataset Overview:
  Shape: (16156, 14)
  Columns: ['Location_Name', 'Located_City', 'Location', 'Location_Type', 'User_ID', 'User_Location', 'User_Locale', 'User_Contributions', 'Travel_Date', 'Published_Date', 'Rating', 'Helpful_Votes', 'Title', 'Text']

 Data Types:
Location_Name         object
Located_City          object
Location              object
Location_Type         object
User_ID               object
User_Location         object
User_Locale           object
User_Contributions     int64
Travel_Date           object
Published_Date        object
Rating                 int64
Helpful_Votes          int64
Title                 object
Text                  object
dtype: object

 Missing Values:
Location_Name         0
Located_City          0
Location              0
Location_Type         0
User_ID               0
User_Location         0
User_Locale           0
User_Contributions    0
Travel_Date           0
Published_Date        0
Rating                0
Helpful_Votes         0
Title

In [4]:


provinces = [
    "Western Province", "Central Province", "Southern Province",
    "Northern Province", "Eastern Province", "North Western Province",
    "North Central Province", "Uva Province", "Sabaragamuwa Province",
]

city_to_district = {
    "Arugam Bay": "Ampara",
    "Ampara": "Ampara",
    "Galle": "Galle",
    "Kalutara": "Kalutara",
    "Trincomalee": "Trincomalee",
    "Matara": "Matara",
    "Colombo": "Colombo",
    "Gampaha": "Gampaha",
    "Nilaveli": "Trincomalee",
    "Kalkudah": "Batticaloa",
    "Nuwara Eliya": "Nuwara Eliya",
    "Kandy": "Kandy",
    "Tissamaharama": "Hambantota",
    "Anuradhapura": "Anuradhapura",
    "Haputale": "Badulla",
    "Badulla": "Badulla",
    "Beruwala": "Kalutara",
    "Jaffna": "Jaffna",
    "Polonnaruwa": "Polonnaruwa",
    "Matale": "Matale",
    "Weligatta": "Hambantota",
    "Ratnapura": "Ratnapura",
    "Tangalle": "Hambantota",
    "Pinnawala": "Kegalle",
    "Deniyaya": "Matara",
    "Koslanda": "Badulla",
    "Mirissa": "Matara",
    "Ella": "Badulla",
    "Negombo": "Gampaha",
    "Sigiriya": "Matale",
    "Batticaloa": "Batticaloa",
    "Bentota": "Galle",
    "Hikkaduwa": "Galle",
    "Mirigama": "Gampaha",
    "Habarana": "Anuradhapura",
}

manual_location_mappings = {
    "Udawalawe National Park": {"province": "Sabaragamuwa Province", "district": "Ratnapura"},
    "North Central Province": {"province": "North Central Province", "district": "Anuradhapura"},
}

province_pattern = re.compile(
    r"(" + "|".join([re.escape(p) for p in provinces]) + r")", flags=re.IGNORECASE
)

def extract_province(location_str):
    """Extract province from location string"""
    if not isinstance(location_str, str) or not location_str.strip():
        return None
    
    loc = location_str.strip()
    
    # Manual mapping
    for k, v in manual_location_mappings.items():
        if k.lower() == loc.lower():
            return v.get("province")
    
    # Pattern matching
    m = province_pattern.search(loc)
    if m:
        return m.group(1)
    
    # Numeric pattern
    m2 = re.search(r"([A-Za-z ]+Province)\b", loc)
    if m2:
        return m2.group(1).strip()
    
    return None

def infer_district(row):
    """Infer district from location and city"""
    city = row.get("Located_City")
    location = row.get("Location")
    
    # Manual mapping
    if isinstance(location, str):
        for k, v in manual_location_mappings.items():
            if k.lower() == location.strip().lower():
                return v.get("district")
    
    # City mapping
    if isinstance(city, str) and city in city_to_district:
        return city_to_district[city]
    
    if not isinstance(location, str):
        return None
    
    parts = [p.strip() for p in location.split(",") if p.strip()]
    
    # Check for explicit district
    for part in parts:
        if "District" in part:
            return part.replace("District", "").strip()
    
    # Infer from position
    if len(parts) >= 3:
        return parts[-2]
    elif len(parts) == 2:
        return parts[0]
    elif len(parts) == 1 and not any(parts[0].lower() == p.lower() for p in provinces):
        return parts[0]
    
    return None

# Apply location extraction
print(" Extracting province and district...")
df["Province"] = df["Location"].apply(extract_province)
df["District"] = df.apply(infer_district, axis=1)

print(f"Province coverage: {df['Province'].notna().sum()}/{len(df)} rows")
print(f"District coverage: {df['District'].notna().sum()}/{len(df)} rows")
print(f"\n Top Provinces:")
print(df["Province"].value_counts().head(10))

 Extracting province and district...
Province coverage: 16156/16156 rows
District coverage: 16156/16156 rows

 Top Provinces:
Province
Central Province          5252
North Central Province    3171
Southern Province         2648
Western Province          1890
Eastern Province          1162
Uva Province              1040
Sabaragamuwa Province      518
Northern Province          475
Name: count, dtype: int64


In [5]:
# Install dependencies if needed
try:
    import pycountry
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pycountry"])
    import pycountry

try:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
    from geopy.exc import GeocoderTimedOut, GeocoderServiceError
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "geopy"])
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
    from geopy.exc import GeocoderTimedOut, GeocoderServiceError

import re

def normalize_location(value):
    """Normalize raw location text into a consistent format."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None

    text = str(value).strip().lower()
    if not text:
        return None

    # Keep useful separators and remove noisy characters
    text = re.sub(r"[^a-z0-9,\s\-.]", " ", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip(" ,")

def build_country_lookup():
    """Build alias -> canonical country map using pycountry + common aliases."""
    alias_map = {}

    for country in pycountry.countries:
        canonical = country.name
        aliases = {country.name, country.alpha_2, country.alpha_3}

        if hasattr(country, "official_name"):
            aliases.add(country.official_name)
        if hasattr(country, "common_name"):
            aliases.add(country.common_name)

        for alias in aliases:
            key = normalize_location(alias)
            if key:
                alias_map[key] = canonical

    # Common user-written shortcuts
    manual_aliases = {
        "usa": "United States",
        "u.s.a": "United States",
        "us": "United States",
        "u.s": "United States",
        "united states of america": "United States",
        "uk": "United Kingdom",
        "u.k": "United Kingdom",
        "uae": "United Arab Emirates",
        "u.a.e": "United Arab Emirates",
        "england": "United Kingdom",
        "scotland": "United Kingdom",
        "wales": "United Kingdom",
        "northern ireland": "United Kingdom",
        "sri lanka": "Sri Lanka",
    }

    for alias, canonical in manual_aliases.items():
        alias_map[normalize_location(alias)] = canonical

    return alias_map

COUNTRY_LOOKUP = build_country_lookup()
COUNTRY_ALIASES_SORTED = sorted(COUNTRY_LOOKUP.keys(), key=len, reverse=True)

def extract_country_direct(normalized_text):
    """Try direct country extraction from normalized location string."""
    if not normalized_text:
        return None

    for alias in COUNTRY_ALIASES_SORTED:
        pattern = rf"(^|[^a-z0-9]){re.escape(alias)}([^a-z0-9]|$)"
        if re.search(pattern, normalized_text):
            return COUNTRY_LOOKUP[alias]
    return None

# Geocoder with gentle rate-limiting for Nominatim policy compliance
geolocator = Nominatim(user_agent="tourism_country_resolution_v1")
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.0,
    max_retries=2,
    error_wait_seconds=2.0,
    swallow_exceptions=False
)

def geocode_country(normalized_text):
    """Geocode location and return detected country if available."""
    try:
        result = geocode(normalized_text, addressdetails=True, language="en")
        if result and isinstance(result.raw, dict):
            address = result.raw.get("address", {})
            country = address.get("country")
            if country:
                return country
    except (GeocoderTimedOut, GeocoderServiceError, Exception):
        return None
    return None

# Parse each unique normalized location once (cache)
df["_normalized_user_location"] = df["User_Location"].apply(normalize_location)

unique_locations = df["_normalized_user_location"].drop_duplicates().tolist()
resolution_cache = {}

for loc in unique_locations:
    if loc is None:
        resolution_cache[loc] = {"country": None, "status": "missing"}
        continue

    direct_country = extract_country_direct(loc)
    if direct_country is not None:
        resolution_cache[loc] = {"country": direct_country, "status": "direct_match"}
        continue

    geo_country = geocode_country(loc)
    if geo_country is not None:
        resolution_cache[loc] = {"country": geo_country, "status": "geocoded"}
    else:
        resolution_cache[loc] = {"country": None, "status": "not_found"}

# Map resolved results back to full dataframe
df["User_Country"] = df["_normalized_user_location"].map(lambda x: resolution_cache[x]["country"])
df["resolution_status"] = df["_normalized_user_location"].map(lambda x: resolution_cache[x]["status"])
df["User_Region"] = None

# Quick quality check
print("Parsing user locations with direct matching + geocoding...")
print(f"Country resolution: {df['User_Country'].notna().sum()}/{len(df)} rows")
print("\nResolution status counts:")
print(df["resolution_status"].value_counts(dropna=False))
print("\nTop 15 Countries:")
print(df["User_Country"].value_counts().head(15))

# Drop helper column used for caching keys
df = df.drop(columns=["_normalized_user_location"], errors="ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 54.8 MB/s eta 0:00:00
Parsing user locations with direct matching + geocoding...
Country resolution: 16112/16156 rows

Resolution status counts:
resolution_status
direct_match    14261
geocoded         1851
not_found          44
Name: count, dtype: int64

Top 15 Countries:
User_Country
United Kingdom          4830
Australia               2101
India                   1689
United States            953
United Arab Emirates     440
Canada                   425
Netherlands              398
Singapore                342
Germany                  329
New Zealand              249
France                   242
Malaysia                 210
Ireland                  150
Switzerland              147
Belgium                  146
Name: count, dtype: int64


In [6]:
# Convert dates
for col in ['Travel_Date', 'Published_Date']:
    df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)
    df[f'{col}_Month'] = df[col].dt.month
    df[f'{col}_Year'] = df[col].dt.year

# Add text features
df['Review_Length'] = df['Text'].apply(len)
df['Title_Length'] = df['Title'].apply(len)

print("Temporal features extracted")
print(f"\n Date Range:")
print(f"  Travel: {df['Travel_Date'].min()} to {df['Travel_Date'].max()}")
print(f"  Published: {df['Published_Date'].min()} to {df['Published_Date'].max()}")

Temporal features extracted

 Date Range:
  Travel: 2010-09-01 00:00:00+00:00 to 2023-05-01 00:00:00+00:00
  Published: 2011-03-12 12:00:09+00:00 to 2023-05-20 05:15:38+00:00


In [7]:
# --- Add Review Delay and Rating Class Columns ---
def get_rating_class(r):
    if r <= 2:
        return "Negative"
    elif r == 3:
        return "Neutral"
    else:
        return "Positive"

df["Rating_Class"] = df["Rating"].apply(get_rating_class)

if "Published_Date" in df.columns and "Travel_Date" in df.columns:
    df["Review_Delay_Days"] = (df["Published_Date"] - df["Travel_Date"]).dt.days
    # Fix negative values (bad data)
    df["Review_Delay_Days"] = df["Review_Delay_Days"].clip(lower=0)
else:
    df["Review_Delay_Days"] = None

print("Added Review_Delay_Days and Rating_Class columns (negative delays fixed)")

Added Review_Delay_Days and Rating_Class columns (negative delays fixed)


In [ ]:
# Drop raw location columns (redundant with extracted features)
df = df.drop(columns=['Location', 'User_Location','User_Region'], errors='ignore')
# Drop 'travel_date' and 'published_date' columns if they exist
for col in ['Travel_Date', 'Published_Date']:
    if col in df.columns:
        df = df.drop(col, axis=1)
        print(f"'{col}' column dropped.")
    else:
        print(f"'{col}' column not found.")
# Drop 'User_ID' column if it exists
if 'User_ID' in df.columns:
    df = df.drop('User_ID', axis=1)
    print("'User_ID' column dropped.")
else:
    print("'User_ID' column not found.")

# Reorder columns as requested
ordered_cols = [
    'Location_Name', 'Located_City', 'Province', 'District', 'Location_Type',
    'User_Locale', 'User_Country', 'resolution_status', 'Travel_Date_Month', 'Travel_Date_Year',
    'Published_Date_Month', 'Published_Date_Year',
    'User_Contributions', 'Rating', 'Helpful_Votes',
    'Title', 'Text', 'Review_Length', 'Title_Length', 'Rating_Class', 'Review_Delay_Days'
   
]
# Only keep columns that exist in the DataFrame
ordered_cols = [col for col in ordered_cols if col in df.columns]
df = df[ordered_cols + [col for col in df.columns if col not in ordered_cols]]

print(f"Cleaned up redundant columns and reordered")
print(f"\nDataset Shape: {df.shape}")
print(f"\nCurrent Columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")
df.to_csv("Processed_Reviews.csv", index=False)

In [ ]:
df["District"].unique().tolist()

In [ ]:
df.head()